# 03 — Gold: Dimensões SCD Tipo 1

Processa 6 dimensões estáveis (SCD1) + DimDate.

**Técnica DuckDB:** `INSERT OR REPLACE INTO` — equivalente ao MERGE (INSERT/UPDATE) do SQL Server.
Requer constraint UNIQUE na natural key (definida no `01_setup.ipynb`).

**Surrogate key:** `CAST(hash(natural_key) % 2147483647 AS INTEGER)` — determinístico entre execuções.

**Destaque:** `DimEmployee` — self-join para hierarquia achatada + `STRING_AGG ORDER BY` para territórios M:N.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# DimEmployee — SCD1 com hierarquia achatada + territórios
# Self-join para ManagerName; STRING_AGG para TerritoryList
# ============================================================

conn.execute("""
    DELETE FROM gold.DimEmployee
""")

conn.execute("""
    INSERT INTO gold.DimEmployee
    WITH emp_hierarchy AS (
        SELECT
            e.EmployeeID,
            e.FirstName || ' ' || e.LastName                   AS FullName,
            e.Title,
            e.HireDate::DATE                                    AS HireDate,
            e.City,
            e.Country,
            e.ReportsTo                                         AS ReportsToID,
            m.FirstName || ' ' || m.LastName                    AS ManagerName
        FROM bronze.employees e
        LEFT JOIN bronze.employees m ON e.ReportsTo = m.EmployeeID
    ),
    territory_agg AS (
        SELECT
            et.EmployeeID,
            -- STRING_AGG com ORDER BY: equivalente ao STRING_AGG ... ORDER BY do SQL Server
            STRING_AGG(TRIM(t.TerritoryDescription), ', ' ORDER BY TRIM(t.TerritoryDescription)) AS TerritoryList,
            FIRST(r.RegionDescription)                          AS RegionName
        FROM bronze.employee_territories et
        LEFT JOIN bronze.territories t  ON et.TerritoryID = t.TerritoryID
        LEFT JOIN bronze.region r       ON t.RegionID     = r.RegionID
        GROUP BY et.EmployeeID
    )
    SELECT
        CAST(hash(CAST(e.EmployeeID AS VARCHAR)) % 2147483647 AS INTEGER) AS EmployeeSK,
        e.EmployeeID,
        e.FullName,
        e.Title,
        e.HireDate,
        e.City,
        e.Country,
        e.ReportsToID,
        e.ManagerName,
        COALESCE(ta.TerritoryList, '')                          AS TerritoryList,
        ta.RegionName,
        current_timestamp                                       AS LoadTimestamp
    FROM emp_hierarchy e
    LEFT JOIN territory_agg ta ON e.EmployeeID = ta.EmployeeID
""")

n = conn.execute("SELECT COUNT(*) FROM gold.DimEmployee").fetchone()[0]
print(f"DimEmployee: {n} linhas (esperado: 9)")

# Preview com hierarquia
print("\nPreview DimEmployee (hierarquia):")
print(conn.execute("""
    SELECT FullName, Title, ManagerName, TerritoryList, RegionName
    FROM gold.DimEmployee ORDER BY FullName
""").fetchdf().to_string(index=False))

DimEmployee: 9 linhas (esperado: 9)

Preview DimEmployee (hierarquia):
        FullName                    Title     ManagerName                                                                                                                  TerritoryList                                         RegionName
   Andrew Fuller    Vice President, Sales             NaN                                                         Bedford, Boston, Braintree, Cambridge, Georgetow, Louisville, Westboro Eastern                                           
  Anne Dodsworth     Sales Representative Steven Buchanan                                                 Bloomfield Hills, Hollis, Minneapolis, Portsmouth, Roseville, Southfield, Troy Northern                                          
 Janet Leverling     Sales Representative   Andrew Fuller                                                                                              Atlanta, Orlando, Savannah, Tampa Southern                            

In [3]:
# ============================================================
# DimCategory — DELETE + INSERT (full refresh SCD1, 8 linhas)
# ============================================================
conn.execute("DELETE FROM gold.DimCategory")
conn.execute("""
    INSERT INTO gold.DimCategory
    SELECT
        CAST(hash(CAST(CategoryID AS VARCHAR)) % 2147483647 AS INTEGER) AS CategorySK,
        CategoryID,
        CategoryName,
        Description,
        current_timestamp AS LoadTimestamp
    FROM bronze.categories
""")
n = conn.execute("SELECT COUNT(*) FROM gold.DimCategory").fetchone()[0]
print(f"DimCategory: {n} linhas (esperado: 8)")

DimCategory: 8 linhas (esperado: 8)


In [4]:
# ============================================================
# DimSupplier — INSERT OR REPLACE (29 linhas)
# ============================================================
conn.execute("""
    INSERT OR REPLACE INTO gold.DimSupplier
    SELECT
        CAST(hash(CAST(SupplierID AS VARCHAR)) % 2147483647 AS INTEGER) AS SupplierSK,
        SupplierID,
        CompanyName,
        City,
        Country,
        current_timestamp AS LoadTimestamp
    FROM bronze.suppliers
""")
n = conn.execute("SELECT COUNT(*) FROM gold.DimSupplier").fetchone()[0]
print(f"DimSupplier: {n} linhas (esperado: 29)")

DimSupplier: 29 linhas (esperado: 29)


In [5]:
# ============================================================
# DimShipper — INSERT OR REPLACE (3 linhas)
# ============================================================
conn.execute("""
    INSERT OR REPLACE INTO gold.DimShipper
    SELECT
        CAST(hash(CAST(ShipperID AS VARCHAR)) % 2147483647 AS INTEGER) AS ShipperSK,
        ShipperID,
        CompanyName,
        Phone,
        current_timestamp AS LoadTimestamp
    FROM bronze.shippers
""")
n = conn.execute("SELECT COUNT(*) FROM gold.DimShipper").fetchone()[0]
print(f"DimShipper: {n} linhas (esperado: 3)")

DimShipper: 3 linhas (esperado: 3)


In [6]:
# ============================================================
# DimTerritory — INSERT OR REPLACE com join em Region
# ============================================================
conn.execute("""
    INSERT OR REPLACE INTO gold.DimTerritory
    SELECT
        CAST(hash(t.TerritoryID) % 2147483647 AS INTEGER) AS TerritorySK,
        t.TerritoryID,
        t.TerritoryDescription,
        t.RegionID,
        r.RegionDescription AS RegionName,
        current_timestamp AS LoadTimestamp
    FROM bronze.territories t
    LEFT JOIN bronze.region r ON t.RegionID = r.RegionID
""")
n = conn.execute("SELECT COUNT(*) FROM gold.DimTerritory").fetchone()[0]
print(f"DimTerritory: {n} linhas (esperado: 53)")

DimTerritory: 53 linhas (esperado: 53)


In [7]:
# ============================================================
# DimDate — gera série 1990-01-01 a 2030-12-31
# DuckDB: generate_series() para criar o range de datas
# ============================================================
existing = conn.execute("SELECT COUNT(*) FROM gold.DimDate").fetchone()[0]

if existing > 0:
    print(f"DimDate já populada: {existing} dias. Pulando.")
else:
    conn.execute("""
        INSERT INTO gold.DimDate
        SELECT
            CAST(strftime(d::DATE, '%Y%m%d') AS INTEGER)    AS DateKey,
            d::DATE                                          AS FullDate,
            YEAR(d)                                          AS Year,
            QUARTER(d)                                       AS Quarter,
            MONTH(d)                                         AS Month,
            STRFTIME(d::DATE, '%B')                          AS MonthName,
            DAY(d)                                           AS Day,
            -- DayOfWeek: 0=Sun, 1=Mon, ..., 6=Sat (DuckDB isodow: 1=Mon, 7=Sun)
            DAYOFWEEK(d)                                     AS DayOfWeek,
            STRFTIME(d::DATE, '%A')                          AS DayName,
            DAYOFWEEK(d) IN (0, 6)                           AS IsWeekend
        FROM generate_series(DATE '1990-01-01', DATE '2030-12-31', INTERVAL '1 day') gs(d)
    """)
    n = conn.execute("SELECT COUNT(*) FROM gold.DimDate").fetchone()[0]
    print(f"DimDate populada: {n} dias (1990-01-01 a 2030-12-31)")

DimDate populada: 14975 dias (1990-01-01 a 2030-12-31)


In [8]:
# ============================================================
# Resumo
# ============================================================
silver_tables = [
    ("gold.DimEmployee",   9),
    ("gold.DimCategory",   8),
    ("gold.DimSupplier",  29),
    ("gold.DimShipper",    3),
    ("gold.DimTerritory", 53),
    ("gold.DimDate",   14976),  # 1990-01-01 a 2030-12-31 = 14976 dias
]

print("\nContagens silver (SCD1 + DimDate):")
for table, expected in silver_tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    ok = n == expected or (table == "gold.DimDate" and n > 14000)
    print(f"  {table:<35} {n:>6} linhas {'OK' if ok else 'FAIL'}")

conn.close()


Contagens silver (SCD1 + DimDate):
  gold.DimEmployee                         9 linhas OK
  gold.DimCategory                         8 linhas OK
  gold.DimSupplier                        29 linhas OK
  gold.DimShipper                          3 linhas OK
  gold.DimTerritory                       53 linhas OK


  gold.DimDate                         14975 linhas OK
